In [3]:
import csv
import os
import math
from pathlib import Path
from typing import List, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchmetrics.detection import MeanAveragePrecision

# =========================
# Configuration
# =========================
DINOV3_GITHUB_LOCATION = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/dinov3"
DINOV3_LOCATION = os.getenv("DINOV3_LOCATION") or DINOV3_GITHUB_LOCATION
DINO_MODEL_NAME = "dinov3_vitl16"
DINO_WEIGHTS = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"

UP_ROOT  = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/uttar_pradesh"
BD_ROOT  = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/bangladesh"
PKP_ROOT = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/pak_punjab"

IMAGE_SIZE    = 800
BATCH_SIZE    = 8
NUM_WORKERS   = 8
NUM_EPOCHS    = 10
BACKBONE_LR   = 1e-5
HEAD_LR       = 1e-4
WEIGHT_DECAY  = 0.04
NUM_CLASSES   = 4  # background + 3 classes (labels 1..3)

BEST_CKPT     = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/checkpoints/best_pak_punjab_val_map50_dinov3.pth"
RESULTS_CSV   = "pak_punjab_region_eval_final.csv"

# =========================
# Dataset
# =========================
class BrickKilnDataset(Dataset):
    """
    Folder layout: <root>/<split>/{images,labels}
    YOLO-OBB line: <cls> x1 y1 x2 y2 x3 y3 x4 y4  (all in [0,1])
    Converted to axis-aligned XYXY for Faster R-CNN.
    """
    IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

    def __init__(self, root: str, split: str, input_size: int = 224):
        self.root = Path(root)
        self.split = split
        cand = self.root if (self.root / "images").is_dir() else (self.root / split)
        self.img_dir = cand / "images"
        self.label_dir = cand / "labels"
        assert self.img_dir.is_dir(), f"Missing images directory: {self.img_dir}"
        assert self.label_dir.is_dir(), f"Missing labels directory: {self.label_dir}"

        self.input_size = int(input_size)
        self.transform = transforms.Compose([
            transforms.Resize((self.input_size, self.input_size),
                              interpolation=transforms.InterpolationMode.BILINEAR,
                              antialias=True),
            transforms.ToTensor(),
        ])

        # Include ALL images (even those with no GT) so FPs are penalized.
        self.img_files: List[str] = sorted(
            [f for f in os.listdir(self.img_dir) if Path(f).suffix.lower() in self.IMG_EXTS]
        )

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx: int):
        img_name = self.img_files[idx]
        img_path = self.img_dir / img_name
        label_path = self.label_dir / f"{Path(img_name).stem}.txt"

        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)
        _, Ht, Wt = img_tensor.shape

        boxes, labels = [], []
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 9:
                        continue
                    cls_id = int(float(parts[0])) + 1  # shift to 1..3 (0 is background)
                    obb = np.array([float(p) for p in parts[1:]], dtype=np.float32)
                    xs = obb[0::2] * Wt
                    ys = obb[1::2] * Ht
                    xmin, ymin = float(np.min(xs)), float(np.min(ys))
                    xmax, ymax = float(np.max(xs)), float(np.max(ys))
                    if xmax > xmin and ymax > ymin:
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(cls_id)

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
        }
        return img_tensor, target


def collate_fn(batch):
    # Keep images even with zero GT boxes (important for correct FP accounting).
    images, targets = list(zip(*batch))
    return list(images), list(targets)

# =========================
# DINOv3 Backbone Wrapper
# =========================
class DinoV3BackboneWrapper(nn.Module):
    """Return {'0': Tensor[B, C, H/16, W/16]} with out_channels=C."""
    def __init__(self, dino_model: nn.Module, patch_stride: int = 16):
        super().__init__()
        self.dino = dino_model
        self.patch_stride = patch_stride
        C = getattr(dino_model, "embed_dim", None) or getattr(dino_model, "num_features", None)
        if C is None:
            with torch.no_grad():
                x = torch.zeros(1, 3, 32, 32)
                tokens, Ht, Wt = self._get_patch_tokens(x)
                C = tokens.shape[-1]
        self.out_channels = C

    @torch.no_grad()
    def _maybe_h_w(self, x):
        _, _, H, W = x.shape
        return math.ceil(H / self.patch_stride), math.ceil(W / self.patch_stride)

    def _get_patch_tokens(self, x):
        try:
            out = self.dino.forward_features(x)
            print("out shape:", type(out), out.keys() if isinstance(out, dict) else out.shape)
            
            if isinstance(out, dict):
                if "x_norm_patchtokens" in out:
                    tokens = out["x_norm_patchtokens"]
                    Ht = out.get("H") or self._maybe_h_w(x)[0]
                    Wt = out.get("W") or self._maybe_h_w(x)[1]
                    print("tokens shape from x_norm_patchtokens:", tokens.shape)
                    print("Ht, Wt:", Ht.shape, Wt.shape)
                    return tokens, Ht, Wt
                if "tokens" in out and out["tokens"] is not None:
                    t = out["tokens"]
                    Ht, Wt = self._maybe_h_w(x)
                    if t.shape[1] == (Ht * Wt + 1):
                        t = t[:, 1:, :]
                    return t, Ht, Wt
            if isinstance(out, torch.Tensor):
                t = out
                Ht, Wt = self._maybe_h_w(x)
                N = Ht * Wt
                if t.shape[1] == N + 1:
                    t = t[:, 1:, :]
                elif t.shape[1] != N:
                    N = t.shape[1]
                    Wt = int(round(math.sqrt(N)))
                    Ht = N // Wt
                return t, Ht, Wt
        except Exception:
            pass

        if hasattr(self.dino, "get_intermediate_layers"):
            t = self.dino.get_intermediate_layers(x, n=1, return_class_token=False)[0]
            Ht, Wt = self._maybe_h_w(x)
            return t, Ht, Wt

        t = self.dino(x)
        Ht, Wt = self._maybe_h_w(x)
        if t.dim() == 3 and t.shape[1] == (Ht * Wt + 1):
            t = t[:, 1:, :]
        return t, Ht, Wt

    def forward(self, x: torch.Tensor):
        tokens, Ht, Wt = self._get_patch_tokens(x)
        print("Tokens shape after _get_patch_tokens:", tokens.shape)
        print("Ht, Wt:", Ht.shape, Wt.shape)
        B, N, C = tokens.shape
        print("Backbone token shape:", tokens.shape)
        feat = tokens.transpose(1, 2).contiguous().view(B, C, Ht, Wt)
        # print("Backbone feature shape:", feat.shape)
        return {"0": feat}

def create_model(dino_model: nn.Module, num_classes: int, image_size: int = 800) -> FasterRCNN:
    backbone = DinoV3BackboneWrapper(dino_model, patch_stride=16)
    # print(f"Backbone out_channels: {backbone.out_channels}")
    print()
    anchor_generator = AnchorGenerator(
        sizes=((16, 32, 64, 128, 256),),
        aspect_ratios=((0.5, 1.0, 2.0),)
    )
    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        min_size=image_size,
        max_size=image_size,
    )
    return model


In [4]:
print(f"DINOv3 location set to {DINOV3_LOCATION}")
dino_model = torch.hub.load(
        repo_or_dir=DINOV3_LOCATION,
        model=DINO_MODEL_NAME,
        source="local",
        weights=DINO_WEIGHTS,
        skip_validation=True,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_model(dino_model, num_classes=NUM_CLASSES, image_size=IMAGE_SIZE).to(device)

DINOv3 location set to /home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/dinov3



In [5]:
model

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=800, mode='bilinear')
  )
  (backbone): DinoV3BackboneWrapper(
    (dino): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (rope_embed): RopePositionEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x SelfAttentionBlock(
          (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): SelfAttention(
            (qkv): LinearKMaskedBias(in_features=1024, out_features=3072, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=

In [7]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# --- Create the model ---
# num_classes = 2 (background + 1 object class)
model = fasterrcnn_resnet50_fpn(weights=None, num_classes=2)
model.eval()

# --- Dummy input: 1 RGB image, 3×800×800 ---
dummy_img = torch.randn(1, 3, 128, 128)

# --- Forward pass ---
with torch.no_grad():
    outputs = model(dummy_img)

# --- Print shapes ---
print("Number of outputs:", len(outputs))
print("Keys in output:", outputs[0].keys())
for k, v in outputs[0].items():
    print(f"{k}: {v.shape}")

"""
Typical output:
Number of outputs: 1
Keys in output: dict_keys(['boxes', 'labels', 'scores'])
boxes: torch.Size([N, 4])
labels: torch.Size([N])
scores: torch.Size([N])
"""

Number of outputs: 1
Keys in output: dict_keys(['boxes', 'labels', 'scores'])
boxes: torch.Size([100, 4])
labels: torch.Size([100])
scores: torch.Size([100])


"\nTypical output:\nNumber of outputs: 1\nKeys in output: dict_keys(['boxes', 'labels', 'scores'])\nboxes: torch.Size([N, 4])\nlabels: torch.Size([N])\nscores: torch.Size([N])\n"

In [8]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Load pretrained Faster R-CNN model
model = fasterrcnn_resnet50_fpn(pretrained=True)
model.eval()

# Create dummy input (batch_size, channels, height, width)
batch_size = 2
dummy_input = torch.rand(batch_size, 3, 800, 600)

print("=" * 60)
print("FASTER R-CNN SHAPE CHECKER")
print("=" * 60)

print(f"\nInput Shape: {dummy_input.shape}")
print(f"  - Batch size: {batch_size}")
print(f"  - Channels: 3 (RGB)")
print(f"  - Height: 800")
print(f"  - Width: 600")

# Run inference
with torch.no_grad():
    outputs = model(list(dummy_input))

print(f"\n{'=' * 60}")
print("OUTPUT SHAPES (per image in batch):")
print("=" * 60)

for i, output in enumerate(outputs):
    print(f"\nImage {i + 1}:")
    print(f"  - Boxes shape: {output['boxes'].shape}")
    print(f"    (num_detections, 4) - [x1, y1, x2, y2] coordinates")
    print(f"  - Labels shape: {output['labels'].shape}")
    print(f"    (num_detections,) - class labels")
    print(f"  - Scores shape: {output['scores'].shape}")
    print(f"    (num_detections,) - confidence scores")
    
    print(f"\n  Number of detections: {len(output['boxes'])}")
    if len(output['boxes']) > 0:
        print(f"  Sample box: {output['boxes'][0].tolist()}")
        print(f"  Sample label: {output['labels'][0].item()}")
        print(f"  Sample score: {output['scores'][0].item():.4f}")

print(f"\n{'=' * 60}")
print("MODEL ARCHITECTURE INFO:")
print("=" * 60)
print(f"Backbone: ResNet50 with FPN (Feature Pyramid Network)")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Optional: Customize number of classes
num_classes = 91  # COCO has 91 classes (including background)
print(f"\nNumber of classes: {num_classes}")
print("(To change: replace the box predictor in model.roi_heads.box_predictor)")

/opt/anaconda3/envs/rishabh_sat/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/rishabh_sat/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FASTER R-CNN SHAPE CHECKER

Input Shape: torch.Size([2, 3, 800, 600])
  - Batch size: 2
  - Channels: 3 (RGB)
  - Height: 800
  - Width: 600

OUTPUT SHAPES (per image in batch):

Image 1:
  - Boxes shape: torch.Size([0, 4])
    (num_detections, 4) - [x1, y1, x2, y2] coordinates
  - Labels shape: torch.Size([0])
    (num_detections,) - class labels
  - Scores shape: torch.Size([0])
    (num_detections,) - confidence scores

  Number of detections: 0

Image 2:
  - Boxes shape: torch.Size([0, 4])
    (num_detections, 4) - [x1, y1, x2, y2] coordinates
  - Labels shape: torch.Size([0])
    (num_detections,) - class labels
  - Scores shape: torch.Size([0])
    (num_detections,) - confidence scores

  Number of detections: 0

MODEL ARCHITECTURE INFO:
Backbone: ResNet50 with FPN (Feature Pyramid Network)
Total parameters: 41,755,286
Trainable parameters: 41,532,886

Number of classes: 91
(To change: replace the box predictor in model.roi_heads.box_predictor)


In [10]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# Load pretrained Faster R-CNN model
model = fasterrcnn_resnet50_fpn(pretrained=True)
print(model)
model.eval()

# Create dummy input (batch_size, channels, height, width)
batch_size = 2
dummy_input = torch.rand(batch_size, 3, 800, 600)

print("=" * 70)
print("FASTER R-CNN LAYER-BY-LAYER SHAPE CHECKER")
print("=" * 70)

print(f"\n[INPUT] Shape: {dummy_input.shape}")
print(f"        (batch_size, channels, height, width)")

# Hook function to capture intermediate shapes
shapes = {}
def get_activation(name):
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            shapes[name] = output.shape
        elif isinstance(output, tuple) and len(output) > 0 and isinstance(output[0], torch.Tensor):
            shapes[name] = output[0].shape
        elif isinstance(output, dict):
            shapes[name] = {k: v.shape if isinstance(v, torch.Tensor) else type(v) for k, v in output.items()}
    return hook

# Register hooks for backbone layers
print(f"\n{'=' * 70}")
print("BACKBONE (ResNet50-FPN):")
print("=" * 70)

backbone_hooks = []
backbone_hooks.append(model.backbone.body.conv1.register_forward_hook(get_activation('conv1')))
backbone_hooks.append(model.backbone.body.layer1.register_forward_hook(get_activation('layer1')))
backbone_hooks.append(model.backbone.body.layer2.register_forward_hook(get_activation('layer2')))
backbone_hooks.append(model.backbone.body.layer3.register_forward_hook(get_activation('layer3')))
backbone_hooks.append(model.backbone.body.layer4.register_forward_hook(get_activation('layer4')))
backbone_hooks.append(model.backbone.fpn.register_forward_hook(get_activation('fpn')))

# Register hooks for RPN
print("\nRPN (Region Proposal Network) hooks registered")
backbone_hooks.append(model.rpn.head.register_forward_hook(get_activation('rpn_head')))

# Run inference
with torch.no_grad():
    # Process images as list for detection model
    images_list = [img for img in dummy_input]
    
    # Get backbone features
    features = model.backbone(dummy_input)
    print(f"\n[BACKBONE OUTPUT] Feature maps at different scales:")
    for key, feat in features.items():
        print(f"  {key}: {feat.shape}")
    
    # Get RPN proposals
    images_tensors = torch.stack(images_list)
    image_sizes = [(800, 600) for _ in range(batch_size)]
    proposals, proposal_losses = model.rpn(images_tensors, features, image_sizes)
    
    print(f"\n[RPN OUTPUT] Proposals (region proposals):")
    for i, prop in enumerate(proposals):
        print(f"  Image {i+1}: {prop.shape} - (num_proposals, 4)")
    
    # Full model output
    outputs = model(images_list)

print(f"\n{'=' * 70}")
print("LAYER SHAPES:")
print("=" * 70)

for name, shape in shapes.items():
    print(f"\n[{name.upper()}]")
    if isinstance(shape, dict):
        for k, v in shape.items():
            print(f"  {k}: {v}")
    else:
        print(f"  Shape: {shape}")

print(f"\n{'=' * 70}")
print("FINAL DETECTION OUTPUT:")
print("=" * 70)

for i, output in enumerate(outputs):
    print(f"\n[IMAGE {i + 1}]")
    print(f"  Boxes:  {output['boxes'].shape} - (num_detections, 4)")
    print(f"  Labels: {output['labels'].shape} - (num_detections,)")
    print(f"  Scores: {output['scores'].shape} - (num_detections,)")
    print(f"  Total detections: {len(output['boxes'])}")
    
    if len(output['boxes']) > 0:
        print(f"\n  Sample detection:")
        print(f"    Box: [{output['boxes'][0][0]:.1f}, {output['boxes'][0][1]:.1f}, "
              f"{output['boxes'][0][2]:.1f}, {output['boxes'][0][3]:.1f}]")
        print(f"    Label: {output['labels'][0].item()}")
        print(f"    Score: {output['scores'][0].item():.4f}")

# Remove hooks
for hook in backbone_hooks:
    hook.remove()

print(f"\n{'=' * 70}")
print("ARCHITECTURE SUMMARY:")
print("=" * 70)
print("1. Input → Conv1 → ResNet Stages (layer1-4) → FPN")
print("2. FPN features → RPN → Region Proposals")
print("3. Proposals + Features → RoI Heads → Box Regression + Classification")
print("4. Output: Bounding boxes, labels, and confidence scores")

/opt/anaconda3/envs/rishabh_sat/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/rishabh_sat/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

AttributeError: 'Tensor' object has no attribute 'tensors'

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# Load pretrained Faster R-CNN model
model = fasterrcnn_resnet50_fpn(pretrained=True)
model.eval()
    
# Create dummy input (batch_size, channels, height, width)
batch_size = 2
input_height = 480  # Change this to your desired height
input_width = 640   # Change this to your desired width
dummy_input = torch.rand(batch_size, 3, input_height, input_width)

print("=" * 70)
print("FASTER R-CNN LAYER-BY-LAYER SHAPE CHECKER")
print("=" * 70)

print(f"\n[ORIGINAL INPUT] Shape: {dummy_input.shape}")
print(f"                 (batch_size, channels, height, width)")
print(f"\nNOTE: Model transform will automatically resize images:")
print(f"      - min_size=800, max_size=1333")
print(f"      - Maintains aspect ratio")

# Hook function to capture intermediate shapes
shapes = {}
def get_activation(name):
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            shapes[name] = output.shape
        elif isinstance(output, tuple) and len(output) > 0 and isinstance(output[0], torch.Tensor):
            shapes[name] = output[0].shape
        elif isinstance(output, dict):
            shapes[name] = {k: v.shape if isinstance(v, torch.Tensor) else type(v) for k, v in output.items()}
    return hook

# Register hooks for backbone layers
print(f"\n{'=' * 70}")
print("BACKBONE (ResNet50-FPN):")
print("=" * 70)

backbone_hooks = []
backbone_hooks.append(model.backbone.body.conv1.register_forward_hook(get_activation('conv1')))
backbone_hooks.append(model.backbone.body.layer1.register_forward_hook(get_activation('layer1')))
backbone_hooks.append(model.backbone.body.layer2.register_forward_hook(get_activation('layer2')))
backbone_hooks.append(model.backbone.body.layer3.register_forward_hook(get_activation('layer3')))
backbone_hooks.append(model.backbone.body.layer4.register_forward_hook(get_activation('layer4')))
backbone_hooks.append(model.backbone.fpn.register_forward_hook(get_activation('fpn')))

# Register hooks for RPN
print("\nRPN (Region Proposal Network) hooks registered")
backbone_hooks.append(model.rpn.head.register_forward_hook(get_activation('rpn_head')))

# Run inference
with torch.no_grad():
    # Process images as list for detection model
    images_list = [img for img in dummy_input]
    
    # Get transformed images to see actual input to backbone
    images, _ = model.transform(images_list)
    print(f"\n[AFTER TRANSFORM] Actual input to backbone:")
    print(f"                  Shape: {images.tensors.shape}")
    print(f"                  (Images resized to maintain aspect ratio)")
    
    # Get backbone features
    features = model.backbone(images.tensors)
    print(f"\n[BACKBONE OUTPUT] Feature maps at different scales:")
    for key, feat in features.items():
        print(f"  {key}: {feat.shape}")
    
    # Get RPN proposals
    image_sizes = [(img.shape[-2], img.shape[-1]) for img in images_list]
    proposals, proposal_losses = model.rpn(images.tensors, features, images.image_sizes)
    
    print(f"\n[RPN OUTPUT] Proposals (region proposals):")
    for i, prop in enumerate(proposals):
        print(f"  Image {i+1}: {prop.shape} - (num_proposals, 4)")
    
    # Full model output
    outputs = model(images_list)

print(f"\n{'=' * 70}")
print("LAYER SHAPES:")
print("=" * 70)

for name, shape in shapes.items():
    print(f"\n[{name.upper()}]")
    if isinstance(shape, dict):
        for k, v in shape.items():
            print(f"  {k}: {v}")
    else:
        print(f"  Shape: {shape}")

print(f"\n{'=' * 70}")
print("FINAL DETECTION OUTPUT:")
print("=" * 70)

for i, output in enumerate(outputs):
    print(f"\n[IMAGE {i + 1}]")
    print(f"  Boxes:  {output['boxes'].shape} - (num_detections, 4)")
    print(f"  Labels: {output['labels'].shape} - (num_detections,)")
    print(f"  Scores: {output['scores'].shape} - (num_detections,)")
    print(f"  Total detections: {len(output['boxes'])}")
    
    if len(output['boxes']) > 0:
        print(f"\n  Sample detection:")
        print(f"    Box: [{output['boxes'][0][0]:.1f}, {output['boxes'][0][1]:.1f}, "
              f"{output['boxes'][0][2]:.1f}, {output['boxes'][0][3]:.1f}]")
        print(f"    Label: {output['labels'][0].item()}")
        print(f"    Score: {output['scores'][0].item():.4f}")

# Remove hooks
for hook in backbone_hooks:
    hook.remove()

print(f"\n{'=' * 70}")
print("ARCHITECTURE SUMMARY:")
print("=" * 70)
print("1. Input → Conv1 → ResNet Stages (layer1-4) → FPN")
print("2. FPN features → RPN → Region Proposals")
print("3. Proposals + Features → RoI Heads → Box Regression + Classification")
print("4. Output: Bounding boxes, labels, and confidence scores")

FASTER R-CNN LAYER-BY-LAYER SHAPE CHECKER

[ORIGINAL INPUT] Shape: torch.Size([2, 3, 480, 640])
                 (batch_size, channels, height, width)

NOTE: Model transform will automatically resize images:
      - min_size=800, max_size=1333
      - Maintains aspect ratio

BACKBONE (ResNet50-FPN):

RPN (Region Proposal Network) hooks registered

[AFTER TRANSFORM] Actual input to backbone:
                  Shape: torch.Size([2, 3, 800, 1088])
                  (Images resized to maintain aspect ratio)

[BACKBONE OUTPUT] Feature maps at different scales:
  0: torch.Size([2, 256, 200, 272])
  1: torch.Size([2, 256, 100, 136])
  2: torch.Size([2, 256, 50, 68])
  3: torch.Size([2, 256, 25, 34])
  pool: torch.Size([2, 256, 13, 17])


AttributeError: 'Tensor' object has no attribute 'tensors'